In [2]:
from model import Transformer
import tensorflow_datasets as tfds


tokenizer = tfds.deprecated.text.SubwordTextEncoder.load_from_file("./tokenizer")


VOCAB_SIZE = tokenizer.vocab_size + 2
NUM_LAYERS = 2
D_MODEL = 256
NUM_HEADS = 8
DFF = 512
DROPOUT = 0.1
model = Transformer(
        vocab_size=VOCAB_SIZE,
        num_layers=NUM_LAYERS,
        dff=DFF,
        d_model=D_MODEL,
        num_heads=NUM_HEADS,
        dropout=DROPOUT
    ).to('cpu')
#mac은 mps


import torch
model_dict = torch.load("./mymodel.pth", map_location='cpu')
model.load_state_dict(model_dict)


import re
def preprocess_sentence(sentence):
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = sentence.strip()
    return sentence
START_TOKEN = [tokenizer.vocab_size]
END_TOKEN = [tokenizer.vocab_size + 1]
VOCAB_SIZE = tokenizer.vocab_size + 2
device = 'cpu'
MAX_LENGTH=40




def evaluate(sentence):
    sentence = preprocess_sentence(sentence)
   
    # 수정 포인트: START_TOKEN과 END_TOKEN을 감싸고 있던 대괄호 [] 제거
    sentence_tensor = torch.tensor(START_TOKEN + tokenizer.encode(sentence) + END_TOKEN).unsqueeze(0).to(device)
    output_tensor = torch.tensor(START_TOKEN).unsqueeze(0).to(device)


    model.eval()
    with torch.no_grad():
        for i in range(MAX_LENGTH):
            predictions = model(sentence_tensor, output_tensor)
           
            # 현재(마지막) 시점의 예측 단어를 가져옵니다.
            predictions = predictions[:, -1:, :]
            predicted_id = torch.argmax(predictions, dim=-1)


            if predicted_id.item() == END_TOKEN[0]:
                break


            output_tensor = torch.cat([output_tensor, predicted_id], dim=-1)


    return output_tensor.squeeze(0).cpu().numpy()


def predict(sentence):
    prediction = evaluate(sentence)
    predicted_sentence = tokenizer.decode(
        [i for i in prediction if i < tokenizer.vocab_size])
   
    print('Input: {}'.format(sentence))
    print('Output: {}'.format(predicted_sentence))
    return predicted_sentence


In [26]:
predict("-")

Input: -
Output: 마음에 드는걸로 하세요


'마음에 드는걸로 하세요'